# Probability as the Foundation of Knowledge Discovery

Your Name

## Getting Started

* Colab - get notebook from gitmystuff DTSC5810 repository
* Save a Copy in Drive
* Remove Copy of
* Edit your name in the cell above and the filename
* Clean up Colab Notebooks folder
* Submit shared link

**How to use this notebook:** This session is narrative-first. The goal is not to master every formula — it's to leave with a working vocabulary and a sense of *why* probability underlies everything else this course will cover. Code cells are here to make ideas concrete and to expose you to the Python tools (`numpy`, `scipy.stats`) you'll keep using all semester. Run every cell, read the explanation before and after it, and don't worry about memorizing syntax.

**By the end of Part 1, you should be able to:**
* Explain where this course's approach to "knowledge discovery" comes from historically (KDD, CRISP-DM)
* Tell the story of how probability grew from gambling into a tool for measuring the natural world
* Recognize a handful of classic probability fallacies when you see them in the wild
* Describe the difference between the Classical, Frequentist, and Bayesian views of probability
* Explain why probability is the *foundation* of statistics
* Read a PMF, a PDF, and a CDF, and know what KDE is estimating


## 1. Why This Matters: We Are Terrible at Randomness

A few years ago, a man won the Spanish national lottery with a ticket ending in the number 48. Proud of his "accomplishment," he explained his method: he had dreamed of the number 7 for seven straight nights, and "7 times 7 is 48."

Anyone with working multiplication tables can spot the error (7 × 7 = 49, not 48). But before we laugh, it's worth noticing *why* the story is funny — and why it's a little uncomfortable. This man did something all of us do constantly: he built a private theory of how the world works, out of pattern and coincidence, and used it to explain an outcome that was, in fact, just noise.

Here's a second example that's less funny, because professionals fall for it too: a doctor tells a patient that a disease test came back positive, and the test is "99% accurate." Most people — including many clinicians — will estimate the odds the patient actually has the disease at somewhere close to 99%. Depending on how rare the disease is, the *real* answer can be closer to 10-20%. We'll come back to exactly why later in this notebook (it's called **base rate neglect**, and it's one of several well-documented probability fallacies humans fall into).

Both stories point at the same idea: **human intuition about probability is not reliable, and that's why formal probability theory exists.** It's the discipline that lets us reason carefully about uncertainty instead of trusting our gut — and reasoning carefully about uncertainty turns out to be the foundation underneath almost everything else in this course: statistics, machine learning, knowledge graphs, causal inference.


## 2. Where This Course Fits: A Short History of "Knowledge Discovery"

The phrase "knowledge discovery" is a direct reference to a formal discipline with its own history. Before we get into probability itself, it's worth knowing where this whole field came from, because probability sits at the center of it.

### KDD — Knowledge Discovery in Databases (1996)

The term "Knowledge Discovery in Databases" (KDD) was formalized by Fayyad, Piatetsky-Shapiro, and Smyth in 1996. It defined knowledge discovery as a *multi-step process*, not a single algorithm:

1. **Selection** — choosing the relevant data
2. **Preprocessing** — cleaning it (missing values, noise, errors)
3. **Transformation** — reshaping it into a usable form (feature engineering)
4. **Data Mining** — applying algorithms to find patterns
5. **Interpretation / Evaluation** — deciding whether the patterns found are actually *knowledge*, or just noise

That last step is the one people skip, and it's the one probability governs. A pattern is only "knowledge" if you can say something about how likely it is to be real rather than coincidental — which is exactly the lottery-guy problem from Section 1, formalized into a pipeline.

### CRISP-DM — Cross-Industry Standard Process for Data Mining

CRISP-DM emerged shortly after KDD and is still, today, the most widely used named methodology in industry data science practice. It covers similar ground but is framed around business needs first: Business Understanding → Data Understanding → Data Preparation → Modeling → Evaluation → Deployment. Where KDD is an academic, discovery-focused framework, CRISP-DM is a practitioner's project-management framework built on top of the same underlying logic.

### SEMMA

SEMMA (Sample, Explore, Modify, Model, Assess) is SAS's own branded version of essentially the same pipeline. You'll mostly encounter it as a historical reference now — it's tied closely to the SAS software ecosystem and has faded as SAS's dominance has faded.

### Why This Matters for a Course Built in the Age of AI

None of these frameworks anticipated dense vector embeddings, knowledge graphs, graph neural networks, or causal discovery — the tools you'll use throughout this semester. But the underlying *logic* of the pipeline hasn't changed: you still have to select and understand data, still have to model it, and still have to ask whether what you found is real. This course is, in a sense, KDD rebuilt for 2026 — the pillars in your syllabus (probability & distributions → vector search → knowledge graphs → graph ML → probabilistic reasoning → causal discovery) are a modern elaboration of that same 30-year-old pipeline.

And notice where probability sits in it: not just in "Data Mining," but underneath *every* stage — it's what lets you decide whether your preprocessing removed noise or signal, whether your model's pattern is real, and whether your interpretation is justified. That's the thesis of this whole session.


In [ ]:
# A simple visual of the KDD pipeline, just to anchor the five stages.
# Don't worry about the plotting code itself -- the point is the diagram it produces.

import matplotlib.pyplot as plt

stages = ["Selection", "Preprocessing", "Transformation", "Data Mining", "Interpretation\n& Evaluation"]
fig, ax = plt.subplots(figsize=(11, 2.5))

for i, stage in enumerate(stages):
    ax.add_patch(plt.Rectangle((i * 2, 0), 1.7, 1, edgecolor='black', facecolor='#dbeafe'))
    ax.text(i * 2 + 0.85, 0.5, stage, ha='center', va='center', fontsize=10, wrap=True)
    if i < len(stages) - 1:
        ax.annotate('', xy=(i * 2 + 2, 0.5), xytext=(i * 2 + 1.7, 0.5),
                     arrowprops=dict(arrowstyle='->', lw=1.5))

ax.set_xlim(-0.3, len(stages) * 2)
ax.set_ylim(-0.3, 1.3)
ax.axis('off')
ax.set_title('The KDD Pipeline (Fayyad et al., 1996)', fontsize=12)
plt.tight_layout()
plt.show()


## 3. A Short History of Quantifying Uncertainty

Probability theory didn't start as a branch of mathematics for its own sake. It started because a gambler had a very specific, very practical complaint.




### 3.1 Games of Chance: Where Probability Was Born

In the mid-1600s, a French nobleman and gambler known as the **Chevalier de Méré** posed a puzzle to the mathematician Blaise Pascal. De Méré had noticed something odd about two bets he liked to make:

* **Bet A:** Roll a single die 4 times. Bet that at least one 6 appears. De Méré knew from experience this bet won slightly more often than it lost — profitable.
* **Bet B:** Roll a pair of dice 24 times. Bet that at least one double-6 appears. De Méré assumed this should be just as profitable, by a simple (but flawed) proportional scaling argument. In practice, it lost money.

Pascal corresponded with **Pierre de Fermat** in 1654 to work out why — and their letters are widely credited as the birth of formal probability theory. Their answer required calculating the probability of an event *not* happening, repeatedly, and subtracting from 1 (the **complement rule**).

Let's verify de Méré's puzzle two ways: by simulation, and by the exact formula Pascal and Fermat would have derived.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=42)

# --- Bet A: at least one 6 in 4 rolls of a single die ---
n_trials = 200_000
rolls_a = rng.integers(1, 7, size=(n_trials, 4))          # 4 rolls per trial
wins_a = np.any(rolls_a == 6, axis=1)                      # did a 6 show up?
p_a_simulated = wins_a.mean()

# --- Bet B: at least one double-6 in 24 rolls of two dice ---
rolls_b1 = rng.integers(1, 7, size=(n_trials, 24))
rolls_b2 = rng.integers(1, 7, size=(n_trials, 24))
double_six = (rolls_b1 == 6) & (rolls_b2 == 6)
wins_b = np.any(double_six, axis=1)
p_b_simulated = wins_b.mean()

print(f"Bet A -- P(at least one 6 in 4 rolls),      simulated: {p_a_simulated:.4f}")
print(f"Bet B -- P(at least one double-6 in 24 rolls), simulated: {p_b_simulated:.4f}")
print()

# The exact complement-rule calculation Pascal and Fermat would have used:
# P(at least one success) = 1 - P(no successes at all)
p_a_exact = 1 - (5/6)**4
p_b_exact = 1 - (35/36)**24

print(f"Bet A -- exact: 1 - (5/6)^4        = {p_a_exact:.4f}")
print(f"Bet B -- exact: 1 - (35/36)^24     = {p_b_exact:.4f}")


Notice Bet A is (barely) a winning bet — just above 50% — and Bet B is (barely) a losing one. De Méré's proportional-scaling intuition was close, but "close" is exactly where gambling intuition and formal probability part ways. This is the **Complement Rule** in action: $P(A) = 1 - P(\text{not } A)$, and it's one of the "Big 5" rules (Complement, Addition, Multiplication, Law of Total Probability, Bayes' Theorem) that essentially everything else in probability is built from.

### 3.2 Pascal's Triangle: A Detour Worth Taking

Pascal's name is attached to one more idea from this era that will resurface all semester: **Pascal's Triangle**, a simple arrangement of numbers where each entry is the sum of the two above it. It looks like a cute arithmetic curiosity, but it's actually a table of **binomial coefficients** — the count of ways to choose $k$ successes out of $n$ trials, written $\binom{n}{k}$. You'll meet this again the moment you touch the Binomial distribution.

Worth knowing: Pascal didn't invent this triangle. It appears independently across history — as the *Meru Prastāra* in India (Pingala, Halayudha), in the work of Al-Karaji and Omar Khayyam in Persia, and in Yang Hui's writings in China, centuries before Pascal. It's a good reminder that mathematical ideas tend to get discovered multiple times, independently, when the underlying need (here: counting combinations) is universal.


In [ ]:
from math import comb

n_rows = 8
for n in range(n_rows):
    row = [comb(n, k) for k in range(n + 1)]
    print(row)

# Each number here is a binomial coefficient: comb(n, k) = "n choose k"
# It answers: "how many different ways can I get k successes out of n trials?"
# This is the exact quantity inside the Binomial distribution's formula,
# which you'll see again later this semester.


### 3.3 From Games to Measurement: Gauss and the Birth of Statistical Error

Probability's second major growth spurt didn't come from gamblers — it came from astronomers.

In 1801, the astronomer Giuseppe Piazzi discovered the dwarf planet **Ceres**, tracked it for about six weeks, and then lost it behind the sun. Astronomers across Europe had only a short, noisy set of position measurements and needed to predict where Ceres would reappear. A 24-year-old **Carl Friedrich Gauss** took on the problem and, in the process, invented the **Method of Least Squares** — a way of finding the "most likely true value" underlying a set of imperfect, noisy measurements, by minimizing the sum of squared errors between predictions and observations:

$$\min \sum (y_i - f(x_i))^2$$

Gauss found Ceres. But the more important legacy is *how* he found it: he showed that if you assume measurement errors are random and symmetric around the truth, the mathematics of "most likely true value" leads directly to the **Normal (Gaussian) distribution** — the bell curve. The normal distribution wasn't originally about naturally-occurring bell-shaped data at all; it fell out of an engineering problem about how to combine noisy observations.

A related figure, **Friedrich Bessel**, later identified something called the "personal equation" — the discovery that different human observers had small, systematic, individually-consistent delays in recording when a star crossed a telescope's crosshairs. This was one of the first formal separations of **random error** (unpredictable noise) from **systematic bias** (a consistent, correctable offset) — a distinction you'll rely on constantly when you evaluate models later this semester.

Here's the throughline worth holding onto: **Pascal and Fermat used probability to settle a bet. Gauss and Bessel used it to find a lost planet and to separate real signal from human error.** In both cases, the tool is the same: a formal way of reasoning about uncertainty. That instinct — guess, measure the error, refine your guess — will reappear almost exactly when we get to the EM algorithm in Part 2.


In [ ]:
# A small least-squares demo in the spirit of Gauss: recovering a "true" line from noisy data.
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
x = np.linspace(0, 10, 30)
true_slope, true_intercept = 2.5, 1.0
noise = rng.normal(loc=0, scale=3, size=x.shape)   # simulate imperfect measurement
y_observed = true_slope * x + true_intercept + noise

# Least squares fit: minimize sum of squared errors, same principle Gauss used
coeffs = np.polyfit(x, y_observed, deg=1)
fitted_slope, fitted_intercept = coeffs

plt.figure(figsize=(7, 4.5))
plt.scatter(x, y_observed, label='Noisy observations', color='gray')
plt.plot(x, true_slope * x + true_intercept, 'g--', label='True underlying relationship')
plt.plot(x, fitted_slope * x + fitted_intercept, 'r', label='Least-squares fit')
plt.legend()
plt.title('Recovering signal from noise: the Gaussian instinct')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

print(f"True line:   y = {true_slope}x + {true_intercept}")
print(f"Fitted line: y = {fitted_slope:.2f}x + {fitted_intercept:.2f}")


## 4. When Intuition Breaks: Three Classic Probability Fallacies




### 4.1 The Monty Hall Problem

Imagine a game show with three doors. Behind one is a car; behind the other two, goats. You pick a door — say, Door 1. The host, who knows what's behind every door, opens a different door — say, Door 3 — revealing a goat. He then asks: **do you want to switch to Door 2, or stay with Door 1?**

Most people's gut instinct is that it doesn't matter — 50/50 either way. That instinct is wrong. Switching wins **2/3** of the time; staying wins only **1/3**. When Marilyn vos Savant published this answer in her *Parade* magazine column in 1990, she received thousands of letters (including from PhD mathematicians) telling her she was mistaken. She wasn't.

The key insight: the host's choice is *not* random. He knows where the car is, and he is constrained to always reveal a goat. That non-random constraint is information, and it's what breaks the naive 50/50 intuition. Let's simulate both strategies over many trials rather than trust intuition.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
n_games = 100_000

doors = np.array([0, 1, 2])
car_positions = rng.integers(0, 3, size=n_games)
initial_picks = rng.integers(0, 3, size=n_games)

stay_wins = 0
switch_wins = 0

for i in range(n_games):
    car = car_positions[i]
    pick = initial_picks[i]

    # host opens a door that is neither the car nor the player's pick
    remaining = [d for d in doors if d != pick and d != car]
    host_opens = remaining[0] if len(remaining) == 1 else rng.choice(remaining)

    # the only other unopened door is the "switch" option
    switch_choice = [d for d in doors if d != pick and d != host_opens][0]

    if pick == car:
        stay_wins += 1
    if switch_choice == car:
        switch_wins += 1

print(f"Stay strategy win rate:   {stay_wins / n_games:.3f}")
print(f"Switch strategy win rate: {switch_wins / n_games:.3f}")


### 4.2 The Conjunction Fallacy: The "Linda Problem"

Psychologists Amos Tversky and Daniel Kahneman posed a famous scenario:

> *Linda is 31, single, outspoken, and very bright. She majored in philosophy. As a student, she was deeply concerned with issues of discrimination and social justice, and participated in anti-nuclear demonstrations.*
>
> *Which is more probable: (a) Linda is a bank teller, or (b) Linda is a bank teller AND is active in the feminist movement?*

The overwhelming majority of people — including many statistically trained people — pick (b). But this is mathematically impossible. Option (b) is a *subset* of option (a): every "bank teller and feminist" is also a "bank teller." A conjunction of two events can never be more probable than either event alone:

$$P(A \cap B) \le P(A)$$

The story is compelling, and representative-sounding stories tend to override this fairly simple rule. This is the **Conjunction Fallacy** — treating a more detailed, specific-sounding story as more probable, when detail necessarily narrows probability rather than expanding it.

### 4.3 The Prosecutor's Fallacy: Real Stakes

Not every probability fallacy is a psychology-lab curiosity. The **Prosecutor's Fallacy** is the error of inverting a conditional probability — treating $P(\text{Evidence} \mid \text{Innocent})$ as if it were the same thing as $P(\text{Innocent} \mid \text{Evidence})$. These are, in general, very different numbers, and confusing them has put real people in prison.

The most widely known example comes from the 1995 **O.J. Simpson** trial. The prosecution presented Simpson's history of domestic violence against his ex-wife, Nicole Brown Simpson, as evidence of guilt. Defense attorney Alan Dershowitz countered that fewer than 0.04% of men who abuse their partners go on to murder them — implying the history of abuse was nearly irrelevant.

That statistic is true, but it answers the wrong question. The real question, given that Nicole Brown Simpson *had been murdered*, was: **what fraction of murdered women who had been battered were killed by their abuser?** That number is dramatically higher — commonly cited estimates put it around 90%. Dershowitz's argument conditioned on the wrong event. Once you correctly condition on the fact of the murder having already occurred, the probability the killer was the abuser is very high, not near-zero.

This is exactly the base-rate/conditioning error from the medical-test example back in Section 1 — the specific numbers change, but the mistake is the same mistake, and it shows up again and again wherever conditional probability meets real-world stakes: courtrooms, medical diagnosis, forensic DNA matching. Getting probability wrong isn't just an academic embarrassment — it has consequences.


**A note before moving on:** there are several more of these documented fallacies worth knowing by name if you want to go deeper on your own — the **Gambler's Fallacy** (believing a random process is "due" for a change after a streak), the **Hot Hand Fallacy** (its cousin — over-crediting streaks with meaning), the **Disjunction Fallacy**, and **Berkson's Paradox** (spurious negative correlations created by selection bias). We won't cover all of them today, but they're worth a search if courtroom statistics, sports analytics, or survey bias interest you.


## 5. Three Ways to Think About Probability

So far we've been using the word "probability" pretty loosely. It's worth pausing to notice that there are actually three distinct philosophical stances on what a probability *is* — and different fields (and different chapters of this course) lean on different ones.

* **Classical (Theoretical) Probability** — probability as a ratio of favorable outcomes to total outcomes, assuming all outcomes are equally likely: $P(E) = \frac{\text{favorable outcomes}}{\text{total outcomes}}$. This is Cardano's and Pascal's world — dice, cards, coins, where you can count outcomes directly.
* **Frequentist (Empirical) Probability** — probability as the long-run relative frequency of an event, estimated by repeating an experiment many times and counting.
* **Bayesian (Subjective/Epistemic) Probability** — probability as a *degree of belief*, which starts at some prior belief and gets updated as new evidence arrives. This is the view underlying Bayes' Theorem, and it's the philosophy behind most modern machine learning uncertainty quantification.

These aren't just academic labels — they lead to genuinely different workflows. A frequentist and a Bayesian analyzing the exact same coin-flip data can report different conclusions, because they're answering subtly different questions ("how often would this data occur under a fair coin?" vs. "given this data, how much should I now believe the coin is fair?").

Let's look at the same simple question — is a coin fair? — through the frequentist and Bayesian lens side by side.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta

rng = np.random.default_rng(3)

# ---- Frequentist view: estimate P(heads) by long-run relative frequency ----
n_flips = 200
flips = rng.integers(0, 2, size=n_flips)   # 1 = heads, 0 = tails
running_estimate = np.cumsum(flips) / np.arange(1, n_flips + 1)

plt.figure(figsize=(7, 4))
plt.plot(running_estimate)
plt.axhline(0.5, color='red', linestyle='--', label='True P(heads) = 0.5')
plt.xlabel('Number of flips')
plt.ylabel('Running estimate of P(heads)')
plt.title('Frequentist view: probability as long-run frequency')
plt.legend()
plt.show()

# ---- Bayesian view: start with a belief (prior), update after seeing data (posterior) ----
# Suppose before seeing any flips, we're unsure whether this coin is fair -- a flat prior.
heads_observed = flips.sum()
tails_observed = n_flips - heads_observed

x = np.linspace(0, 1, 200)
prior = beta.pdf(x, 1, 1)                                  # flat prior: "no strong belief yet"
posterior = beta.pdf(x, 1 + heads_observed, 1 + tails_observed)  # updated after seeing the data

plt.figure(figsize=(7, 4))
plt.plot(x, prior, label='Prior belief (before data)')
plt.plot(x, posterior, label='Posterior belief (after 200 flips)')
plt.axvline(0.5, color='red', linestyle='--', alpha=0.5)
plt.xlabel('P(heads)')
plt.ylabel('Density')
plt.title('Bayesian view: probability as belief, updated by evidence')
plt.legend()
plt.show()

print(f"Observed: {heads_observed} heads, {tails_observed} tails out of {n_flips} flips")


Notice what happened: the frequentist plot just tracks a running count and converges toward the truth as more data arrives (this convergence, by the way, is itself a formal result called the **Law of Large Numbers**, which we'll name properly in the next section). The Bayesian plot instead shows a whole *distribution of belief* narrowing and shifting as evidence accumulates — starting flat (no opinion) and sharpening around 0.5 once the data comes in.

The blue line represents the density (height = $1.0$) of the prior distribution; the region enclosed underneath the blue line is what represents the total area equal to $1.0$.

After 200 flips, the orange line represents about 11 times more probability density at that specific peak compared to the prior density of the blue line.

You'll meet the Bayesian view again explicitly later this semester when we build Bayesian Belief Networks — it's the same core idea, just applied to more complex, connected variables instead of a single coin.


## 6. The Thesis: Probability Is the Foundation of Statistics

We now have enough pieces on the table to state this module's central claim plainly.

**Probability** and **statistics** are often used interchangeably in casual speech, but they are, formally, *inverse* operations:

* **Probability** reasons **deductively**, from a known population or model *forward* to expected data. *"If this die is fair, what fraction of 4-roll sequences will contain at least one 6?"* — this is exactly the de Méré calculation from Section 3.
* **Statistics** reasons **inductively**, from observed sample data *backward* to an unknown population or model. *"I rolled a die 100 times and got a 6 only 4 times — is this die actually fair?"*

Probability answers come first, historically and logically, because statistics *depends on* probability to work at all. Every statistical method you'll use this semester — confidence intervals, hypothesis tests, machine learning model evaluation — is, underneath, an application of probability theory to the reverse-inference problem: given this data, what can I responsibly claim about the process that generated it?

Two theorems formally bridge these two directions, and you'll see both by name repeatedly this semester:

* **The Law of Large Numbers (LLN)** — as your sample size grows, a sample average converges to the true population value. This is exactly what you watched happen in the frequentist coin-flip plot above.
* **The Central Limit Theorem (CLT)** — regardless of the shape of the underlying population distribution, the *distribution of sample means* drawn from it tends toward a Normal distribution as sample size grows. This is one of the most important, almost eerie results in all of statistics — it's *why* the normal distribution shows up everywhere, even when the raw data doesn't look normal at all.

Let's see the CLT directly: we'll draw samples from a distribution that is deliberately *not* normal (skewed, lopsided) and watch the distribution of sample *means* become bell-shaped anyway.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)

# Start with a deliberately skewed, non-normal population (exponential distribution)
population = rng.exponential(scale=2.0, size=100_000)

sample_size = 40
n_samples = 5000
sample_means = [rng.choice(population, size=sample_size, replace=True).mean()
                for _ in range(n_samples)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(population, bins=60, color='indianred')
axes[0].set_title('The population itself: heavily skewed')
axes[0].set_xlabel('Value')

axes[1].hist(sample_means, bins=60, color='steelblue')
axes[1].set_title(f'Distribution of {n_samples} sample means (n={sample_size} each)')
axes[1].set_xlabel('Sample mean')

plt.suptitle('The Central Limit Theorem: skewed population, normal-shaped sample means')
plt.tight_layout()
plt.show()


The population on the left is nothing like a bell curve. But the sample *means* on the right are — reliably, almost regardless of what the original population looked like, as long as the sample size is reasonably large. This is why so much of classical statistics can safely lean on the normal distribution even when real-world data rarely looks normal on its own: it's not the raw data that needs to be normal, it's the *sampling distribution of the statistic* you're computing from it.

This is the bridge: probability describes how populations behave; the LLN and CLT are what let us walk that bridge *backward*, from a sample back to trustworthy claims about a population — which is the entire enterprise of statistics, and by extension, of data-driven knowledge discovery.


## What is not Probability

### 1. Occurrences / All Possible Occurrences

$$\text{Theoretical Probability} = \frac{\text{Number of Favorable Outcomes}}{\text{Total Number of Possible Outcomes}}$$

* **What it measures:** The formal, combinatorial definition of probability under the assumption of **equally likely outcomes** (also known as *classical probability*).
* **Condition:** You must know the full sample space $\Omega$ ahead of time without running an experiment.
* **Die Example:** A standard six-sided die has 6 possible outcomes ($\{1, 2, 3, 4, 5, 6\}$). The event "rolling a 1" is 1 outcome out of 6.

$$P(1) = \frac{1}{6} \approx 0.1667$$


* **Key Property:** This is a fixed, deductive value—it doesn't depend on actually rolling the die.


### 2. Occurrences / Events That Have Occurred

$$\text{Empirical Estimate (Relative Frequency)} = \frac{\text{Number of Observed Occurrences}}{\text{Total Number of Observed Events (Trials)}}$$

* **What it measures:** An **inductive estimate** based on collected sample data.
* **Condition:** Requires running an experiment $n$ times and recording what actually happened.
* **Die Example:** You roll the die $n = 10$ times and observe a 1 four times ($k = 4$).

$$\hat{p}(1) = \frac{4}{10} = 0.40$$


* **Key Property:** This value fluctuates with every sample run due to sampling variability.


### Summary Comparison

| Metric | Formula | Nature | Context |
| --- | --- | --- | --- |
| **Theoretical Probability ($P$)** | $\frac{\text{Favorable Outcomes}}{\text{All Possible Outcomes}}$ | Deductive / Exact | Known physical system or mathematical model |
| **Empirical Estimate ($\hat{p}$)** | $\frac{\text{Target Events Observed}}{\text{Total Events Occurred}}$ | Inductive / Sample-based | Real-world experiments and data collection |

As the number of observed events ($n$) approaches infinity, the ratio of **Occurrences / Events Occurred** converges to the **Occurrences / All Possible Occurrences** ratio via the **Law of Large Numbers**.

## Probability Foundations

* The probability that two events will both occur can never be greater than the probability that each will occur individually. Why not? Simple arithmetic: the chances that event A will occur = the chances that events A and B will occur + the chance that event A will occur and event B will not occur.   
* If two possible events, A and B, are independent, then the probability that both A and B will occur is equal to the product of their individual probabilities
* If an event can have a number of different and distinct possible outcomes, A, B, C, and so on, then the probability that either A or B will occur is equal to the sum of the individual probabilities of A and B, and the sum of the probabilities of all the possible outcomes (A, B, C, and so on) is 1 (that is, 100%)

## The "AND" & "OR" Relationship

Add when you want to find the probability of **either** event happening, but they cannot happen at the same time (they are mutually exclusive). Look for the word "OR"

### The Rule

$$P(A \text{ or } B) = P(A) + P(B)$$

$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

### The Example

What is the probability of rolling a 3 **OR** a 5 on a single six-sided die?

* **Step 1:** Find the individual probabilities.
* Probability of rolling a 3: $P(3) = \frac{1}{6}$
* Probability of rolling a 5: $P(5) = \frac{1}{6}$


* **Step 2:** Add them together because of the word **OR**.


In [ ]:
# code has been added
from fractions import Fraction

# Sample space for a fair 6-sided die
sample_space = {1, 2, 3, 4, 5, 6}
n_total = len(sample_space)

# Define events
event_3 = {3}
event_5 = {5}

# Individual probabilities
p_3 = Fraction(len(event_3), n_total)  # 1/6
p_5 = Fraction(len(event_5), n_total)  # 1/6

# Union of mutually exclusive events: P(A or B) = P(A) + P(B)
p_3_or_5 = p_3 + p_5

print(f"Exact Probability: {p_3_or_5}")
print(f"Decimal Probability: {float(p_3_or_5):.4f}")

**Result:** **33.3%** ($\frac{1}{3}$)

Multiply when you want to find the probability of two events occurring **together** or **in sequence**. Look for the word "AND"

### The Rule

$$P(A \text{ and } B) = P(A) \times P(B)$$

This expression represents the **Intersection** of two events.


**Intersection ($\cap$, AND):** Represents the probability that both events occur together ($P(A \cap B)$). The general multiplication formula for any two events is:

$$P(A \cap B) = P(A) \cdot P(B \mid A)$$

When $A$ and $B$ are **independent** (the outcome of $A$ does not affect the probability of $B$, so $P(B \mid A) = P(B)$), the intersection formula simplifies to:

$$P(A \cap B) = P(A) \times P(B)$$


**Union ($\cup$, OR):** In contrast, the union uses addition ($P(A \cup B) = P(A) + P(B) - P(A \cap B)$) to find the probability that at least one of the events occurs.

### The Example

What is the probability of getting a **Heads** from a coin flip, **AND** then rolling a die and getting a **6**?

* **Step 1:** Find the individual probabilities.
* Probability of flipping Heads: $P(\text{Heads}) = \frac{1}{2}$
* Probability of rolling a 6: $P(6) = \frac{1}{6}$
* **Step 2:** Multiply them together because of the word **AND**.


In [ ]:
# code has been added
from fractions import Fraction

# Sample spaces
coin_outcomes = {'H', 'T'}
die_outcomes = {1, 2, 3, 4, 5, 6}

# Individual probabilities
p_heads = Fraction(1, len(coin_outcomes))  # 1/2
p_six = Fraction(1, len(die_outcomes))      # 1/6

# Intersection of independent events: P(A and B) = P(A) * P(B)
p_heads_and_six = p_heads * p_six

print(f"Exact Probability: {p_heads_and_six}")
print(f"Decimal Probability: {float(p_heads_and_six):.4f}")

**Result:** **8.3%**

Summary

* If you see **OR**, your pool of winning outcomes gets *larger*, so you **ADD** the chances.
* If you see **AND**, you are forcing multiple conditions to happen together, making it *harder* to win, so you **MULTIPLY** the fractions together (which makes the final percentage smaller).

## 7. One Last Building Block: A Quick Look at Single-Variable Distributions

Everything so far has been about probability conceptually and historically. Before we hand off to Part 2 — where we'll go from one variable to *many* variables at once — let's make sure we have a shared, working vocabulary for describing a single random variable's distribution. If this is familiar, treat it as a fast refresher; if some of it is new, don't worry about mastering it today.

* **Random variable** — a variable whose value is the numerical outcome of a random process (a die roll, a coin flip, someone's height).
* **PMF (Probability Mass Function)** — for a *discrete* random variable (countable outcomes, like a die roll), the PMF gives the exact probability of each possible value.
* **PDF (Probability Density Function)** — for a *continuous* random variable (like height, or time), the PDF gives relative density, not probability directly — for continuous variables, the probability of any *exact* single value is technically zero; only ranges (areas under the curve) have nonzero probability.
* **CDF (Cumulative Distribution Function)** — the probability that the variable takes a value *less than or equal to* some number $x$. It's the running total, the area accumulated so far.

Let's see a PMF (discrete) and a PDF/CDF (continuous) side by side using `scipy.stats` — a library you'll be using constantly this semester.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- PMF: Binomial(n=10, p=0.5) -- e.g. "10 coin flips, how many heads?" ---
n, p = 10, 0.5
x_discrete = np.arange(0, n + 1)
axes[0].bar(x_discrete, stats.binom.pmf(x_discrete, n, p), color='slateblue')
axes[0].set_title('PMF: Binomial(n=10, p=0.5)')
axes[0].set_xlabel('Number of heads')
axes[0].set_ylabel('Probability')

# --- PDF: Normal(mu=0, sigma=1) ---
x_cont = np.linspace(-4, 4, 500)
axes[1].plot(x_cont, stats.norm.pdf(x_cont, loc=0, scale=1), color='seagreen')
axes[1].fill_between(x_cont, stats.norm.pdf(x_cont, loc=0, scale=1), alpha=0.2, color='seagreen')
axes[1].set_title('PDF: Normal(0, 1)')
axes[1].set_xlabel('x')
axes[1].set_ylabel('Density')

# --- CDF: same Normal(0, 1) ---
axes[2].plot(x_cont, stats.norm.cdf(x_cont, loc=0, scale=1), color='darkorange')
axes[2].set_title('CDF: Normal(0, 1)')
axes[2].set_xlabel('x')
axes[2].set_ylabel('Cumulative probability')

plt.tight_layout()
plt.show()


## KDE
Notice the PDF and CDF are directly related: the CDF at any point is just the *accumulated area* under the PDF up to that point. This relationship — density in, cumulative probability out — is one you'll use constantly, whether you're computing a p-value, a percentile, or a confidence interval later this semester.

What if you don't know the distribution's shape ahead of time?

Every example above assumed we already knew we were dealing with a Binomial or a Normal distribution and just needed its parameters. Real data is rarely so polite. **Kernel Density Estimation (KDE)** is a *non-parametric* way to estimate a distribution's shape directly from data, without assuming it belongs to any named family at all — it works by placing a small "bump" (kernel) at every data point and summing them into a smooth curve.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(5)

# Deliberately create data that does NOT come from one clean, simple distribution --
# it's actually a blend of two normal distributions stitched together.
group_a = rng.normal(loc=0, scale=1, size=1000)
group_b = rng.normal(loc=6, scale=1.5, size=600)
messy_data = np.concatenate([group_a, group_b])

plt.figure(figsize=(8, 4.5))
sns.histplot(messy_data, kde=True, color='gray', bins=40)
plt.title('KDE on messy, two-lump data: no single named distribution fits this well')
plt.xlabel('Value')
plt.show()


Look closely at that shape: two distinct "bumps." No single named distribution from the classical toolkit (Normal, Binomial, Poisson...) is going to fit this well, because it isn't *one* distribution — it's a blend of two. KDE traced the shape faithfully anyway, without ever being told that.

That bump-within-a-bump shape is exactly where **Part 2** picks up. What do you do when your data isn't cleanly one distribution, but several overlapping ones? What happens when you have not just one variable like this, but ten, all correlated with each other in complicated ways? Those are the questions behind **Gaussian Mixture Models**, the **EM algorithm**, and **Copulas** — and they're the direct on-ramp to this module's graded activity: estimating multivariate density functions and fitting parametric copulas on real, messy, tabular data.


# Addendums

## Gambling

Starting around the 1500s

* **Math Context**
  * **Foundations of Base-10:** Indian mathematicians developed the base-10 positional system, notation, and the concept of zero between the 1st and 7th centuries. Though transmitted to the West around 1200 by Islamic scholars, it didn't fully replace Roman numerals until the 1500s.
  * **The Hindu-Arabic System:** Our modern system uses ten symbols $\{0,1,2,3,4,5,6,7,8,9\}$, where place values scale exponentially by powers of ten ($10^0, 10^1, 10^2$).
  * **Operational Notation:** The plus ($+$) and minus ($-$) signs were introduced by German merchants and mathematicians at the turn of the 16th century.
  * **The Equals Sign (1557):** Invented by Robert Recorde, who chose two parallel lines because *"no two things can be more equal."*
  * **The Scientific Revolution:** The 1500s mark the deep rooting of modern science, bookended by monumental breakthroughs in astronomy (Copernicus) and anatomy (Vesalius) in 1543.
* **Gerolamo Cardano** (24 September 1501– 21 September 1576), The Book on Games of Chance was not a popular character due to his writings (medical practice was a sham, blood letting, burning, drilling holes in the head, etc)
    * Mother tried to abort him
    * Caught the bubonic plague
    * Children were a challenge (daughter got pregnant with brother Giovanni, had an abortion, and became infertile, Giovanni was executed for supected murder of his wife)
    * Son 2 liked killing small animals at an early age and eventually became a torturer for the Inquisition, who later gave evidence of heresy that imprisoned his father
    * Cardano used gambling to pay for his medical school
    * An example of Cardano's writing: Suppose a random process has many equally likely outcomes, some favorable (that is, winning), some unfavorable (losing). Then the probability of obtaining a favorable outcome is equal to the proportion of outcomes that are favorable. The set of all possible outcomes is called the sample space
    * After he was freed, Cardano moved to Rome, where he received a lifetime annuity from Pope Gregory XIII (after first having been rejected by Pope Pius V, who died in 1572) and finished his autobiography. He was accepted in the Royal College of Physicians, and as well as practising medicine he continued his philosophical studies until his death in 1576.
* **Galileo Galilei** (15 February 1564 – 8 January 1642)
    * Introduces the idea that science must focus on experience and experimentation—how nature operates—rather than on what intuition dictates or our minds find appealing. And most of all, it must be done with mathematics.
    * Wrote a book Thoughts on the Game of Dice
    * Why does 10 show up more often than 9 when rolling 3 die?
    * Logic says the dice should sum to 10 and 9 with equal frequency: both 10 and 9 can be constructed in 6 ways from the throw of three dice. For 9 we can write those ways as (621), (531), (522), (441), (432), and (333). For 10 they are (631), (622), (541), (532), (442), and (433).
    * But consider the outcome (631) consists of the possibilities (1,3,6), (1,6,3), (3,1,6), (3,6,1), (6,1,3), and (6,3,1), whereas the outcome (333) consists only of (3,3,3). Once we’ve made this decomposition, we can see that the outcomes are equally probable and we can apply the law. Since there are 27 ways of rolling a 10 with three dice but only 25 ways to get a total of 9, Galileo concluded that with three dice, rolling a 10 was 27/25, or about 1.08, times more likely
    * Dies under 'house arrest' for heresy
* **Chevalier de Méré’s, Antoine Gombaud**
    * Antoine Gombaud (widely known by his pen name, the **Chevalier de Méré**) had a problem in 1654 because his mathematical intuition about gambling was failing him at the tables, costing him a significant amount of money.
    * Gombaud had figured out a highly profitable wager: betting that he could roll at least one **6** in **4 rolls** of a single die. His baseline logic, a logic which Cardano also held, was that since the probability of rolling a six is $1/6$, rolling four times meant a success rate of $4 \times (1/6) = 2/3$ (or roughly 66.7%).
    * Buoyed by this success, he scaled the bet up to two dice. He reasoned that since the odds of rolling a double-six are $1/36$, he should scale his attempts by the same factor. He wagered that he could roll at least one **double-six** in **24 rolls** of a pair of dice, expecting the exact same $24 \times (1/36) = 2/3$ chance of success.
    * To his frustration, Gombaud consistently lost money on the second bet. He was unable to comprehend why two seemingly identical mathematical proportions behaved entirely differently. Gombaud thought this was a linear problem.
    * Gombaud consulted some friends - Pierre de Fermat and Blaise Pascal. They proved that while linear expectation scales, the probability of an event occurring over multiple trials is nonlinear because "not happening" is compounded.
    * What's the problem? If rolling a single die 4 times yields a $4 \times \frac{1}{6} = \frac{2}{3}$ chance of getting a six, then rolling it 6 times would mean a $6 \times \frac{1}{6} = 100\%$ guarantee of a six. Rolling it 7 times would mean a $116.7\%$ chance.
    * For 1 die, the probability of getting at least one four - $$\text{P(At least one 6)} = 1 - \text{P(No 6s at all)}$$ is about 51.77%
    * What's the probability of rolling four sixes? $(1/6)^4 = \frac{1}{1296}$
    * What's the probability of rolling exactly one six in four rolls? To find the probability of rolling **exactly one 6 in 4 rolls**, we have to look at the problem slightly differently than before.
    * If we want *exactly one* 6, it means we must roll a 6 once, and a non-6 (a 1, 2, 3, 4, or 5) on the other three rolls.
    * This is a classic **Binomial Probability** problem, which requires two steps: calculating the odds of one specific sequence, and then multiplying it by the number of ways that sequence can happen.
    * Let's say we roll the 6 on the very first try, and fail on the next three. The probability of that specific sequence (**6, Not, Not, Not**) is: $$\frac{1}{6} \times \frac{5}{6} \times \frac{5}{6} \times \frac{5}{6} = \frac{125}{1,296}$$
    * But since we have four rolls we have to multiply the probability by 4:  $\text{P(Exactly one 6)} = 4 \times \left( \frac{125}{1,296} \right) = \frac{500}{1,296} = \frac{125}{324}$ which is $\approx 0.3858$ or 38.58%
      * Way 1: $[6, \text{Not}, \text{Not}, \text{Not}]$ $\rightarrow$ Probability is $\frac{125}{1,296}$
      * Way 2: $[\text{Not}, 6, \text{Not}, \text{Not}]$ $\rightarrow$ Probability is $\frac{125}{1,296}$
      * Way 3: $[\text{Not}, \text{Not}, 6, \text{Not}]$ $\rightarrow$ Probability is $\frac{125}{1,296}$
      * Way 4: $[\text{Not}, \text{Not}, \text{Not}, 6]$ $\rightarrow$ Probability is $\frac{125}{1,296}$
    * Rolling a double-six in 24 tries is only about 49.1% ($1 - (35/36)^{24}$).   
* **Blaise Pascal** (19 June 1623 – 19 August 1662) added to probability and gambling
  * Pascal's development of probability theory was his most influential contribution to mathematics. Originally applied to gambling, today it is extremely important in economics, especially in actuarial science
  * From this discussion, the notion of expected value was introduced
* **Pascal's Triangle:** Pascal’s triangle is useful any time you need to know the number of ways in which you can choose some number of objects from a collection that has an equal or greater number - https://en.wikipedia.org/wiki/Pascal%27s_triangle
* **1996 World Series**
  * For example (The Drunkard's Walk): 1996 World Series (Braves vs Yankees). Atlanta was up 2 - 0
  * **The Sample Space:** With a maximum of 5 games left to play, and 2 possible outcomes per game (Yankees win $Y$, or Braves win $B$), there are $2^5 = 32$ total equally likely future sequences.
  * **The Yankees' Path to Victory:** To win the series, the Yankees must win **at least 4** of the remaining 5 games.
  * **Exactly 4 wins:** Can happen in **5 ways** ($BYYYY, YBYYY, YYBYY, YYYBY, YYYYB$)
  * **Exactly 5 wins:** Can happen in **1 way** ($YYYYY$)
  * **Total Yankees Wins:** $5 + 1 = 6 \text{ ways out of } 32 \approx 18.75\%$
  * **The Braves' Path to Victory:** The Braves win if the Yankees fail to get 4 wins. This means the Yankees win either 3, 2, 1, or 0 games.
  * **Total Braves Wins:** $10 + 10 + 5 + 1 = 26 \text{ ways out of } 32 \approx 81.25\%$

**The Pascal's Triangle Shortcut Using the Magic Number (4)**
* **1** way for the Yankees to win 0 games (Braves win the series)
* **5** ways for the Yankees to win 1 game (Braves win the series)
* **10** ways for the Yankees to win 2 games (Braves win the series)
* **10** ways for the Yankees to win 3 games (Braves win the series)
* **5** ways for the Yankees to win 4 games (**Yankees win the series**)
* **1** way for the Yankees to win 5 games (**Yankees win the series**)

*(Historical Postscript: Despite having only an 18.75% mathematical chance, the Yankees defied the odds and won the next 4 games straight to take the series!)*
* **Pascal's Death**
    * In 1662, a few days after Pascal died, a servant noticed a curious bulge in one of Pascal’s jackets. The servant pulled open the lining to find hidden within it folded sheets of parchment and paper. Pascal had apparently carried them with him every day for the last eight years of his life. Scribbled on the sheets, in his handwriting, was a series of isolated words and phrases dated November 23, 1654. The writings were an emotional account of the trance, in which he described how God had come to him and in the space of two hours delivered him from his corrupt ways. Following that revelation, Pascal had dropped most of his friends, calling them “horrible attachments.”
    * He sold his carriage, his horses, his furniture, his library—everything except his Bible. He gave his money to the poor, leaving himself with so little that he often had to beg or borrow to obtain food. He wore an iron belt with points on the inside so that he was in constant discomfort and pushed the belt’s spikes into his flesh whenever he found himself in danger of feeling happy. He denounced his studies of mathematics and science. Of his childhood fascination with geometry, he wrote, “I can scarcely remember that there is such a thing as geometry. I recognize geometry to be so useless…it is quite possible I shall never think of it again.”
    * Yet Pascal remained productive. In the years that followed the trance, he recorded his thoughts about God, religion, and life. Those thoughts were later published in a book titled Pensées, a work that is still in print today. And although Pascal had denounced mathematics, amid his vision of the futility of the worldly life is a mathematical exposition in which he trained his weapon of mathematical probability squarely on a question of theology and created a contribution just as important as his earlier work on the problem of points.

The Drunkard's Walk



## The Binomial Context

### **$n$ = Number of Trials**

* **Definition:** The total number of independent events, attempts, or games being observed in the experiment.
* **In the World Series Example:** **$n = 5$** (There are a maximum of 5 remaining games left to play).

### **$k$ = Number of Successes**

* **Definition:** The specific number of successful outcomes you are trying to calculate the probability for. This is the variable that changes depending on the question you ask.
* **In the World Series Example:** If you want to know the probability of the Yankees winning exactly 4 games, **$k = 4$**. (To find their total chance of winning the series, you would calculate the equation for $k = 4$ and $k = 5$, then add them together).

### **$p$ = Probability of Success on a Single Trial**

* **Definition:** The baseline probability of a "success" happening in any individual, isolated trial. This value must stay constant across all $n$ trials.
* **In the World Series Example:** **$p = 0.5$** (Assuming both teams are perfectly matched, the Yankees have a 50% chance of winning any single game).
* *Note:* The counterpart to $p$ is often written as **$q$** or **$(1-p)$**, which represents the probability of *failure* on a single trial ($1 - 0.5 = 0.5$).

When you plug these parameters into the binomial equation, they explicitly isolate every moving part of the problem:

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

If we look at the Yankees trying to win exactly 4 out of 5 games:

$$\begin{aligned}
P(X = 4) &= \binom{5}{4} \times (0.5)^4 \times (0.5)^{5-4} \\
&= \binom{5}{4} \times (0.5)^4 \times (0.5)^1
\end{aligned}$$

* **$\binom{5}{4}$** tells you *how many paths* result in 4 wins (from Pascal's Triangle).
* **$(0.5)^4$** is the probability of winning those 4 games.
* **$(0.5)^1$** is the probability of losing the remaining 1 game.

## Fraternal Twins

Suppose a mother is carrying fraternal twins and wants to know the odds of having two girls, a boy and a girl, and so on.

* The sample space consists of all the possible lists of the sexes of the children in their birth order: (girl, girl), (girl, boy), (boy, girl), and (boy, boy)
* Same with 2 coin toss (HH, TT, HT, TH)
* Situation in which one problem is another in disguise: isomorphism
* Two girls: 25 percent.
* The chance that at least one of the babies will be a girl is the chance that both will be girls plus the chance that just one will be a girl—that is, 25 percent plus 50 percent, which is 75 percent.
* What are the chances, given that one of the children is a girl, that both children will be girls? One might reason this way: 50 percent?
* Although the statement of the problem says that one child is a girl, it doesn’t say which one, and that changes things. *
* The new information—one of the children is a girl—means that we are eliminating from consideration the possibility that both children are boys. So eliminate (boy, boy) from the sample space.
* That leaves only 3 outcomes in the sample space: (girl, boy), (boy, girl), and (girl, girl). Of these, only (girl, girl) is the favorable outcome—that is, both children are daughters—so the chances that both children are girls is 1 in 3, or
33 percent.
* For instance, if the problem had asked for the chances of both children being girls given that the first child is a girl, then we would have eliminated both (boy, boy) and (boy, girl) from the sample space and the odds would have been 1 in 2, or 50 percent

## The Monty Hall Problem

The **Monty Hall Problem** is perhaps the most famous, brain-twisting example of how human intuition fails when confronted with conditional probability.

Its popularity—and the fierce debate surrounding it—is inextricably tied to **Marilyn vos Savant**, who famously held the Guinness World Record for the highest recorded IQ and wrote the "Ask Marilyn" column in *Parade* magazine.

---

### 1. The Setup: The Game Show Scenario

The problem is loosely based on the American TV game show *Let’s Make a Deal*, hosted by Monty Hall:

1. You are presented with **3 closed doors**.
2. Behind one door is a **brand-new car**; behind the other two are **goats**.
3. You pick a door—say, **Door 1**. (You don't open it yet).
4. The host, Monty Hall—**who knows what is behind every door**—opens one of the remaining doors (say, **Door 3**), revealing a goat.
5. Monty then turns to you and asks: *"Do you want to switch to Door 2?"*

> **The Question:** Is it to your advantage to switch your choice, or does it not matter because there is now a 50/50 chance either way?

---

### 2. Marilyn vos Savant’s Solution (1990)

In her September 9, 1990 column, a reader submitted this exact question. Marilyn answered simply and correctly:

> **Yes, you should switch. Switching doubles your chances of winning from $1/3$ to $2/3$.**

The public response was immediate, intense, and overwhelmingly hostile.

Marilyn received over **10,000 letters**, including roughly **1,000 from PhDs, university professors, and mathematicians**, telling her she was completely wrong. Common intuition insists that with two unopened doors left, the probability must be $50/50$ ($1/2$).

### Quotes from the Experts at the Time:

* *"You blew it, and you blew it big! Since the host shows you a goat, there is now a 1/2 chance of getting the car. As a professional mathematician, I am very concerned with the general public's lack of mathematical skills. Please confess your mistake!"*
— **Dr. Robert Sachs**, George Mason University
* *"May I suggest that the next time you encounter a problem of this nature, you consult a textbook on probability before publishing the answer?"*
— **Dr. Charles Reid**, University of Florida
* *"You are the goat!"*
— Another disgruntled reader.

Despite the condescending backlash, Marilyn held her ground and published follow-up columns providing visual aids, frequency tables, and simulations until the mathematical community was forced to run computer simulations—and publicly apologize.

---

### 3. Why Intuition Fails (The 50/50 Trap)

People fall into the $50/50$ trap because they treat Monty’s action as random. If a random gust of wind blew open Door 3 revealing a goat, the remaining two doors *would* indeed be $50/50$.

However, **Monty’s behavior is not random**. Monty operates under strict constraints:

1. He must open an unchosen door.
2. He must **never** reveal the car.
3. He must always reveal a goat.

Monty's specialized knowledge **injects non-random information into the system**.

---

### 4. The Math: Breaking Down the Probabilities

When you make your initial selection, there are two mutually exclusive possibilities:

```
                          [ INITIAL PICK ]
                                 │
           ┌─────────────────────┴─────────────────────┐
           ▼                                           ▼
   Case A: Picked a Goat                       Case B: Picked the Car
   Probability = 2/3                           Probability = 1/3
           │                                           │
           ▼                                           ▼
   Monty IS FORCED to open                     Monty can open EITHER
   the ONLY remaining goat door.               remaining goat door.
           │                                           │
           ▼                                           ▼
   The remaining closed door                   The remaining closed door
   MUST contain the Car.                       contains a Goat.
           │                                           │
           ▼                                           ▼
   SWITCHING WINS!                             SWITCHING LOSES!

```

### The Breakdown Table

| Actual Behind Doors (1, 2, 3) | Your Pick | Door Monty Must Open | Result if You SWITCH |
| --- | --- | --- | --- |
| 🚗 **Car**, 🐐 Goat, 🐐 Goat | Door 1 | Door 2 or 3 | **Loses** (Gets Goat) |
| 🐐 Goat, 🚗 **Car**, 🐐 Goat | Door 1 | Door 3 (Forced) | **WINS (Gets Car)** |
| 🐐 Goat, 🐐 Goat, 🚗 **Car** | Door 1 | Door 2 (Forced) | **WINS (Gets Car)** |

* **If you Stay:** You win only if you picked correctly on step 1 $\implies P(\text{Win}) = \mathbf{1/3}$.
* **If you Switch:** You win whenever you picked a goat on step 1 $\implies P(\text{Win}) = \mathbf{2/3}$.

---

### 5. The Extreme Scale Intuition Trick

If the 3-door scenario still feels counter-intuitive, Marilyn suggested expanding the scale:

Imagine there are **1,000 doors**:

1. You pick **Door 1**. Your odds of having the car are $1/1000$.
2. The remaining 999 doors collectively hold a $999/1000$ chance of having the car.
3. Monty walks down the line and opens **998 doors**, revealing 998 goats, leaving only **Door 743** closed.

Monty intentionally bypassed Door 743 out of 999 options. Do you stick with your initial $1/1000$ guess on Door 1, or switch to Door 743?

The choice becomes glaringly obvious: Monty's intentional filtering concentrated all the probability mass of those 999 doors straight into **Door 743**.

---

### 6. Formal Proof via Bayes' Theorem

Let $C_i$ be the event that the car is behind door $i$, and $M_k$ be the event that Monty opens door $k$.

Suppose you pick **Door 1** ($P(C_1) = P(C_2) = P(C_3) = 1/3$), and Monty opens **Door 3** ($M_3$):

We want to find $P(C_2 \mid M_3)$ (the probability the car is behind Door 2 given Monty opened Door 3):

$$P(C_2 \mid M_3) = \frac{P(M_3 \mid C_2) \cdot P(C_2)}{P(M_3)}$$

1. **Likelihoods:**
* $P(M_3 \mid C_1) = 1/2$ (If car is behind 1, Monty can open 2 or 3 randomly).
* $P(M_3 \mid C_2) = 1$ (If car is behind 2, Monty is **forced** to open 3).
* $P(M_3 \mid C_3) = 0$ (Monty cannot reveal the car).


2. **Law of Total Probability for $P(M_3)$:**

$$P(M_3) = \left(\frac{1}{2} \cdot \frac{1}{3}\right) + \left(1 \cdot \frac{1}{3}\right) + \left(0 \cdot \frac{1}{3}\right) = \frac{1}{6} + \frac{1}{3} = \frac{1}{2}$$


3. **Posterior Calculation:**

$$P(C_2 \mid M_3) = \frac{1 \cdot \frac{1}{3}}{\frac{1}{2}} = \frac{2}{3}$$



---

### Legacy of the Controversy

Even legendary mathematician **Paul Erdős**—one of the most prolific mathematicians in history—refused to believe Marilyn's solution until he was shown a Monte Carlo computer simulation generating the $2/3$ win rate over thousands of trials.

Marilyn vos Savant's story remains a landmark lesson in intellectual humility, proving that human intuition—no matter how educated—is easily misled by conditional constraints.